In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np

# Configuration
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2' # Standard, fast sentence transformer
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Running on: {DEVICE}")

Running on: cuda


In [2]:
class SemanticRouter:
    def __init__(self, advisor_map, model_name=MODEL_NAME):
        """
        Initialize the router and precompute advisor embeddings (Step 2).
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(DEVICE)
        self.advisors = list(advisor_map.keys())
        self.descriptions = list(advisor_map.values())
        
        # --- Step 2: Encode Advisor Descriptions (Precomputed Once) ---
        print("Precomputing advisor embeddings...")
        with torch.no_grad():
            self.advisor_embeddings = self._encode(self.descriptions)
        print(f"Cached embeddings for {len(self.advisors)} advisors.")

    def _encode(self, texts):
        """
        Helper: Encodes a list of texts into normalized embeddings using Mean Pooling.
        Implements: E = M_gating(Text)
        """
        # Tokenize
        inputs = self.tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
        
        # Forward pass
        outputs = self.model(**inputs)
        
        # Mean Pooling - Take attention mask into account for correct averaging
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = inputs['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
        
        # Sum embeddings and divide by sum of mask to get mean
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        
        # Normalize embeddings (Critical for Cosine Similarity)
        # If vectors are normalized, Cosine Similarity == Dot Product
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        return embeddings

    def route(self, query, top_k=2, temperature=0.1):
        """
        Executes the routing algorithm for a single query.
        """
        # --- Step 1: Encode Query ---
        with torch.no_grad():
            E_Q = self._encode([query]) # Shape: [1, hidden_dim]

        # --- Step 3: Compute Similarity Scores ---
        # CosSim(A, B) = dot(A_norm, B_norm)
        # S shape: [1, num_advisors]
        S = torch.mm(E_Q, self.advisor_embeddings.T) 
        
        # --- Step 4: Softmax Normalization (with Temperature) ---
        # Formula: W_i = Softmax(S_i / tau)
        scaled_scores = S / temperature
        W = F.softmax(scaled_scores, dim=1)
        
        # Convert to readable format
        weights_np = W.cpu().numpy().flatten()
        scores_np = S.cpu().numpy().flatten()
        
        # --- Step 5: Top-K Selection ---
        # Get indices of top k weights
        top_k_indices = np.argsort(weights_np)[::-1][:top_k]
        
        results = []
        for idx in top_k_indices:
            results.append({
                "Advisor": self.advisors[idx],
                "Raw_Sim_Score": scores_np[idx],
                "Gating_Weight": weights_np[idx]
            })
            
        return results, weights_np

In [3]:
advisor_profiles = {
    "Diagnosis_Advisor": 
        "Identifies possible diseases and medical conditions based on reported symptoms, clinical patterns, and medical knowledge graph relationships.",

    "Treatment_Advisor": 
        "Recommends appropriate treatments, medications, management plans, and therapeutic options based on diagnosed medical conditions and clinical guidelines.",

    "Doctor_Recommendation_Advisor": 
        "Suggests the most suitable medical specialist or healthcare provider based on the patient’s symptoms, suspected condition, and required level of care."
}


In [5]:
# Initialize Router
router = SemanticRouter(advisor_profiles)

# Test Queries
test_queries = [

    # # Diagnosis-focused queries
    # "I have been experiencing chest pain and shortness of breath for two days.",
    # "Severe headache with sensitivity to light and nausea.",
    # "Persistent dry cough and mild fever for a week.",
    # "Sudden weakness on one side of my body and difficulty speaking.",
    # "I feel constant fatigue and unexplained weight loss.",

    # # Treatment-focused queries
    # "What medications are usually prescribed for high blood pressure?",
    # "How can I treat recurring migraines effectively?",
    # "Best treatment options for eczema flare-ups?",
    # "What are the management steps for type 2 diabetes?",
    # "How is chronic back pain typically treated?",

    # # Doctor recommendation-focused queries
    # "Which specialist should I consult for ongoing heart palpitations?",
    # "What kind of doctor handles nerve pain in the legs?",
    # "Should I see a dermatologist for severe acne scars?",
    # "Who do I consult for persistent digestive problems?",
    # "What doctor treats frequent dizziness and balance issues?",

    # # Mixed-intent queries (good for routing stress test)
    # "I have chest tightness, what could it be and which doctor should I see?",
    # "My child has a high fever and rash, what should I do?",
    # "I’ve been having panic attacks, do I need medication or therapy?",
    # "Severe knee pain after running, is this serious?",
    # "Blurred vision and headache, should I visit a neurologist?"
    # "I'm a 25 year old female. I have fever,chills and constant coughing. could this indicate a specific illness affecting my respiratory system?"
    "i am feeling sweating, trembling, shortness of breath and dizziness. please provide a diagnosis for me."
]


# Run and Display
print("\n" + "="*60)
print(f"{'QUERY ROUTING RESULTS':^60}")
print("="*60)

for q in test_queries:
    print(f"\nUser Query: '{q}'")
    
    # Use a low temperature (0.05) to simulate sharp routing (Step 4 note)
    top_advisors, all_weights = router.route(q, top_k=2, temperature=0.05)
    
    # Create a small dataframe for the Top-K results
    df = pd.DataFrame(top_advisors)
    print(df.to_string(index=False))
    print("-" * 60)

Precomputing advisor embeddings...
Cached embeddings for 3 advisors.

                   QUERY ROUTING RESULTS                    

User Query: 'i am feeling sweating, trembling, shortness of breath and dizziness. please provide a diagnosis for me.'
                      Advisor  Raw_Sim_Score  Gating_Weight
            Diagnosis_Advisor       0.301403       0.461374
Doctor_Recommendation_Advisor       0.297230       0.424433
------------------------------------------------------------
